# 03. 실험 1 — 파라미터 복원 실험

앰프 계열 5 × 정답 θ 4 = 가짜 곡 20개 (EGDB test DI + 믹스 증강).
비교: 시작점(prior_center 사전 범위 중심 / prior_multi 중심+샘플 / encoder 인코더 θ̂) × 통계(장기 평균 스펙트럼 / +레벨별 / 4종) × 세션(없음 / 정확 / 틀림), 기준 (b) 세션만 경로.
결과는 `runs/exp1/exp1_results.jsonl`에 한 줄씩 쌓여서 중단 후 다시 실행하면 이어 간다.

In [ ]:
# 공통 준비: Drive 연결, 저장소 받기, 패키지 설치
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DRIVE = '/content/drive/MyDrive/song2nam'   # 데이터·실행 기록·체크포인트를 둘 Drive 폴더
REPO = '/content/song2nam'
BRANCH = 'main'                               # 작업 브랜치로 바꿔서 쓸 수 있음
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/gmeag01/song2nam.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)
# data/, runs/를 Drive에 연결해서 세션이 끊겨도 남게 한다
for d in ['data', 'runs']:
    if os.path.islink(d):
        continue
    subprocess.run(['rm', '-rf', d])
    os.symlink(f'{DRIVE}/{d}', d)

In [ ]:
# Colab에는 torch/torchaudio가 이미 있으므로 나머지만 설치
!grep -vE '^(torch|torchaudio)$' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import os
from ampcap.synth_data import di_reader_for_split
from ampcap.evaluate import Exp1Config, run_experiment1, summarize_exp1
EGDB, SPLIT = 'data/EGDB/audio_DI', 'data/egdb_split.json'
test_r = di_reader_for_split(EGDB, 'test', SPLIT)
pool_r = di_reader_for_split(EGDB, 'pool', SPLIT)
from ampcap.encoder import BEST_PATH, try_load
enc = try_load(BEST_PATH, 'cuda')   # 없거나 θ 버전이 다르면 None (사전 범위 시작점)
cfg = Exp1Config(steps=600)
# 시간이 부족하면 조건을 줄인다(설계 문서 8장: 실험 1의 일부 조건부터 뺀다)
# cfg.stat_sets = {'all4': ('ltas', 'level', 'flat', 'dyn')}

In [ ]:
res = run_experiment1(test_r, pool_r, 'runs/exp1', cfg, encoder_model=enc, device='cuda')

In [ ]:
df = summarize_exp1('runs/exp1/exp1_results.jsonl')
df.to_csv('runs/exp1/summary.csv'); df

## 끝까지 지키는 비교: 시작점 비교, 세션 정보 유무 비교

In [ ]:
import pandas as pd, json
rows = [json.loads(l) for l in open('runs/exp1/exp1_results.jsonl', encoding='utf-8')]
d = pd.DataFrame([{k: v for k, v in r.items() if k != 'theta_err'} for r in rows])
a = d[d.stats.isin(['all4', '-'])].pivot_table(index='start', columns='session', values='esr', aggfunc='median')
a.to_csv('runs/exp1/start_vs_session_esr.csv'); a